# Previsão da carga de energia elétrica com séries temporais e IA generativa

**O caso:** a Aurora Energia, uma comercializadora fictícia, planeja em outubro os contratos de compra de energia
dos 12 meses seguintes. A referência é a **carga mensal do Sistema Interligado Nacional (SIN)**, publicada pelo ONS.

Este notebook tem quatro partes:

1. **Exploração da série:** tendência, sazonalidade, quebras e relação com a economia.
2. **Previsão e comparação de modelos:** estatísticos, machine learning e modelos de fundação (transformers).
3. **Agente analista:** um LLM aberto do Hugging Face com ferramentas, RAG e skills.
4. **O LLM pode melhorar a previsão?** Variáveis de eventos extraídas de texto e o LLM como previsor.

**Dados:** carga de energia diária do ONS (dados abertos, 2000 em diante), somada nos quatro subsistemas e agregada por mês,
e o IBC-Br, índice de atividade econômica do Banco Central (série 24363 do SGS).

Antes de começar: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("AULA_SMOKE") == "1"   # teste automatizado em CPU, com modelos menores
REPO = "https://github.com/pgcferreira-stack/aula-series-llm-fgv"

if IN_COLAB:
    !pip uninstall -q -y torchao
    !pip install -q statsforecast mlforecast chronos-forecasting timesfm smolagents holidays "transformers>=4.51"
    if not os.path.exists("aula-series-llm-fgv"):
        !git clone -q --depth 1 {REPO}
    os.chdir("aula-series-llm-fgv")

import torch   # carregar o torch antes do LightGBM evita conflito de bibliotecas em alguns sistemas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
os.makedirs("resultados", exist_ok=True)
print("Device:", DEVICE)

# Parte 1 — Exploração da série

## 1.1 Os dados

In [ ]:
dados = pd.read_csv("dados/carga_sin_mensal.csv", parse_dates=["mes"]).set_index("mes")
carga = dados["carga_mwmed"]
print(f"{len(carga)} meses, de {carga.index.min():%m/%Y} a {carga.index.max():%m/%Y}")
dados.tail()

*Para refazer o arquivo a partir da fonte:* o ONS publica um CSV por ano em
`https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_<ano>.csv`.
Somamos os quatro subsistemas por dia e tiramos a média de cada mês completo.

## 1.2 A série inteira e os eventos

In [ ]:
EVENTOS = [
    ("Racionamento", "2001-06-01", "2002-02-01"),
    ("Crise 2008", "2008-10-01", "2009-06-01"),
    ("Recessão", "2015-01-01", "2016-12-01"),
    ("Pandemia", "2020-03-01", "2020-08-01"),
    ("MMGD / calor", "2023-05-01", "2023-12-01"),
]
fig, eixo = plt.subplots(figsize=(14, 5))
eixo.plot(carga.index, carga / 1000, color="#14213D", lw=1.5)
eixo.plot(carga.index, carga.rolling(12, center=True).mean() / 1000, color="#E8833A", lw=2, label="média móvel de 12 meses")
for nome, ini, fim in EVENTOS:
    eixo.axvspan(pd.Timestamp(ini), pd.Timestamp(fim), color="#2F6690", alpha=0.15)
    eixo.text(pd.Timestamp(ini), carga.max() / 1000 * 1.01, nome, fontsize=9, rotation=0)
eixo.set_ylabel("mil MWmed"); eixo.set_title("Carga mensal do SIN"); eixo.legend(loc="lower right")
plt.show()

## 1.3 Crescimento ano a ano

In [ ]:
anual = carga.groupby(carga.index.year).mean()
anual = anual[carga.groupby(carga.index.year).size() == 12]   # só anos completos
cresc = anual.pct_change() * 100

fig, eixos = plt.subplots(1, 2, figsize=(14, 4))
eixos[0].bar(anual.index, anual / 1000, color="#2F6690"); eixos[0].set_title("Carga média anual (mil MWmed)")
cores = ["#B8561B" if v < 0 else "#2F6690" for v in cresc.fillna(0)]
eixos[1].bar(cresc.index, cresc, color=cores); eixos[1].axhline(0, color="black", lw=0.8)
eixos[1].set_title("Crescimento sobre o ano anterior (%)")
plt.show()

yoy = carga.pct_change(12) * 100
print(f"Crescimento médio anual 2000–{anual.index.max()}: {((anual.iloc[-1] / anual.iloc[0]) ** (1 / (len(anual) - 1)) - 1) * 100:.1f}% ao ano")
print("Maiores quedas sobre o mesmo mês do ano anterior:")
print(yoy.nsmallest(5).round(1).to_string())

## 1.4 Sazonalidade: o padrão dentro do ano

Cada linha é um ano. Para comparar anos com níveis diferentes, dividimos cada mês pela média do seu ano.

In [ ]:
tab = carga.to_frame("carga").reset_index(drop=True)
tab["ano"], tab["mes"] = carga.index.year, carga.index.month
tab["indice"] = tab.carga / tab.groupby("ano").carga.transform("mean")
completos = tab.groupby("ano").carga.transform("size") == 12
tab = tab[completos]

fig, eixos = plt.subplots(1, 2, figsize=(14, 4))
mapa = plt.cm.viridis
anos = sorted(tab.ano.unique())
for i, ano in enumerate(anos):
    s = tab[tab.ano == ano]
    eixos[0].plot(s.mes, s.carga / 1000, color=mapa(i / len(anos)), lw=1)
eixos[0].set_title("Carga por mês, uma linha por ano (mais claro = mais recente)"); eixos[0].set_xticks(range(1, 13))
tab.boxplot(column="indice", by="mes", ax=eixos[1]); eixos[1].set_title("Índice sazonal (mês ÷ média do ano)")
plt.suptitle(""); plt.show()

indice = tab.groupby("mes").indice.mean()
print("Índice sazonal médio por mês:")
print((indice * 100 - 100).round(1).rename(lambda m: f"mês {m:02d}").to_string())

## 1.5 Decomposição STL: tendência + sazonalidade + resíduo

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(carga, period=12, robust=True).fit()
fig, eixos = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
for eixo, (nome, s) in zip(eixos, [("Série", carga), ("Tendência", stl.trend), ("Sazonalidade", stl.seasonal), ("Resíduo", stl.resid)]):
    eixo.plot(s.index, s / 1000, color="#14213D", lw=1); eixo.set_ylabel(nome)
eixos[0].set_title("Decomposição STL (mil MWmed)"); plt.tight_layout(); plt.show()

forca_tendencia = max(0, 1 - stl.resid.var() / (stl.trend + stl.resid).var())
forca_sazonal = max(0, 1 - stl.resid.var() / (stl.seasonal + stl.resid).var())
print(f"Força da tendência: {forca_tendencia:.2f} | força da sazonalidade: {forca_sazonal:.2f}  (0 = nenhuma, 1 = máxima)")
print(f"Amplitude sazonal típica: ±{stl.seasonal.abs().mean():,.0f} MWmed ({stl.seasonal.abs().mean() / carga.mean() * 100:.1f}% da carga média)")

residuo_pct = (stl.resid / carga * 100)
print("\nMeses com maior resíduo (fora do padrão de tendência + sazonalidade):")
print(residuo_pct.abs().nlargest(8).index.strftime("%m/%Y").tolist())

## 1.6 Autocorrelação: a série tem memória?

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, eixos = plt.subplots(1, 3, figsize=(15, 3.5))
plot_acf(carga, lags=36, ax=eixos[0], title="ACF da série original")
plot_acf(carga.diff(12).dropna(), lags=36, ax=eixos[1], title="ACF após diferença de 12 meses")
plot_pacf(carga.diff(12).dropna(), lags=36, ax=eixos[2], title="PACF após diferença de 12 meses", method="ywm")
plt.tight_layout(); plt.show()

## 1.7 Estacionariedade: teste ADF

Hipótese nula do teste ADF: a série **tem raiz unitária** (não é estacionária). p-valor abaixo de 0,05 rejeita a hipótese nula.

In [ ]:
from statsmodels.tsa.stattools import adfuller

versoes = {
    "série original": carga,
    "diferença de 12 meses": carga.diff(12),
    "diferença de 1 e de 12 meses": carga.diff(12).diff(1),
}
for nome, s in versoes.items():
    estat, p = adfuller(s.dropna(), autolag="AIC")[:2]
    print(f"{nome:32} estatística ADF = {estat:6.2f}   p-valor = {p:.3f}   {'estacionária' if p < 0.05 else 'NÃO estacionária'}")

## 1.8 Carga e atividade econômica

In [ ]:
ibc = dados["ibc_br"]
comp = pd.DataFrame({"carga_yoy": carga.pct_change(12) * 100, "ibc_yoy": ibc.pct_change(12) * 100}).dropna()
periodo = comp.loc["2004":"2022"].drop(pd.date_range("2020-03-01", "2021-06-01", freq="MS"), errors="ignore")
elasticidade, intercepto = np.polyfit(periodo.ibc_yoy, periodo.carga_yoy, 1)
corr = periodo.corr().iloc[0, 1]

fig, eixos = plt.subplots(1, 2, figsize=(14, 4))
eixos[0].plot(comp.index, comp.carga_yoy, label="carga", color="#14213D")
eixos[0].plot(comp.index, comp.ibc_yoy, label="IBC-Br", color="#E8833A")
eixos[0].set_title("Crescimento sobre o mesmo mês do ano anterior (%)"); eixos[0].legend()
eixos[1].scatter(periodo.ibc_yoy, periodo.carga_yoy, s=10, color="#2F6690")
x = np.linspace(periodo.ibc_yoy.min(), periodo.ibc_yoy.max(), 10)
eixos[1].plot(x, intercepto + elasticidade * x, color="#B8561B")
eixos[1].set_xlabel("IBC-Br (% a/a)"); eixos[1].set_ylabel("carga (% a/a)"); eixos[1].set_title("2004–2022, sem a pandemia")
plt.show()
print(f"Correlação: {corr:.2f} | elasticidade: cada 1 p.p. de crescimento do IBC-Br ≈ {elasticidade:.2f} p.p. na carga")
print("\nCrescimento médio a/a por período (%):")
print(comp.groupby(np.where(comp.index >= "2023-05-01", "a partir de mai/2023", "até abr/2023")).mean().round(1).to_string())

# Parte 2 — Previsão e comparação de modelos

**Tarefa:** prever os **12 meses seguintes**. Para comparar modelos de forma honesta usamos **backtesting**:
voltamos no tempo, treinamos só com o que se sabia até aquela data, prevemos 12 meses e comparamos com o que aconteceu.
Repetimos em três pontos de corte diferentes.

In [ ]:
import holidays

H = 12
ULTIMO = carga.index.max()
ORIGENS = [ULTIMO - pd.DateOffset(months=H * k) for k in (3, 2, 1)]
print("Pontos de corte:", [f"{o:%m/%Y}" for o in ORIGENS], "| último mês com dado:", f"{ULTIMO:%m/%Y}")

feriados = holidays.Brazil(years=range(2000, ULTIMO.year + 3))
def dias_uteis(mes):
    dias = pd.date_range(mes, mes + pd.offsets.MonthEnd(0))
    return int(sum(d.weekday() < 5 and d not in feriados for d in dias))

def calendario(meses):
    """Variáveis conhecidas de antemão para qualquer mês, passado ou futuro."""
    meses = pd.DatetimeIndex(meses)
    return pd.DataFrame({"unique_id": "SIN", "ds": meses, "mes_ano": meses.month,
                         "dias_uteis": [dias_uteis(m) for m in meses]})

serie = pd.DataFrame({"unique_id": "SIN", "ds": carga.index, "y": carga.values})
def futuro(origem):
    return pd.date_range(origem + pd.DateOffset(months=1), periods=H, freq="MS")

## 2.1 Modelos estatísticos (statsforecast)

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive, AutoETS, AutoARIMA, AutoTheta

prev = []
t0 = time.time()
for origem in ORIGENS:
    treino = serie[serie.ds <= origem]
    sf = StatsForecast(models=[SeasonalNaive(season_length=12), AutoETS(season_length=12),
                               AutoARIMA(season_length=12), AutoTheta(season_length=12)], freq="MS")
    p = sf.forecast(df=treino, h=H, level=[80])
    p["origem"] = origem
    prev.append(p)
prev_stats = pd.concat(prev)
print(f"{time.time() - t0:.1f}s")
prev_stats.head()

In [ ]:
sf_final = StatsForecast(models=[AutoETS(season_length=12), AutoARIMA(season_length=12)], freq="MS").fit(serie)
print("Modelo ETS escolhido:", sf_final.fitted_[0, 0].model_["method"])
print("Modelo ARIMA escolhido:", sf_final.fitted_[0, 1].model_["arma"], "(p, q, P, Q, período, d, D)")

## 2.2 Machine learning: LightGBM com defasagens (mlforecast)

In [ ]:
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences

def lightgbm_backtest(colunas_extra=None):
    colunas_extra = colunas_extra or {}
    saidas = []
    for origem in ORIGENS:
        treino = serie[serie.ds <= origem].merge(calendario(serie.ds[serie.ds <= origem]), on=["unique_id", "ds"])
        fut = calendario(futuro(origem))
        for nome, f in colunas_extra.items():
            treino[nome], fut[nome] = f(treino.ds), f(fut.ds)
        mlf = MLForecast(
            models={"LightGBM": lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=15,
                                                  min_child_samples=5, n_jobs=1, verbose=-1)},
            freq="MS", lags=[1, 2, 3, 12], target_transforms=[Differences([12])])
        mlf.fit(treino, static_features=[])
        p = mlf.predict(H, X_df=fut)
        p["origem"] = origem
        saidas.append(p)
    return pd.concat(saidas)

t0 = time.time()
prev_lgb = lightgbm_backtest()
print(f"{time.time() - t0:.1f}s")

## 2.3 Modelos de fundação: Chronos-2 e TimesFM 2.5 (zero-shot)

Os dois modelos foram pré-treinados em milhões de séries temporais e **não são treinados com os nossos dados**:
recebem o histórico como contexto e devolvem a previsão.

In [ ]:
from chronos import Chronos2Pipeline

chronos = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)

def chronos_backtest(colunas_extra=None, nome="Chronos-2"):
    colunas_extra = colunas_extra or {}
    saidas = []
    for origem in ORIGENS:
        ctx = serie[serie.ds <= origem].copy()
        fut = None
        if colunas_extra:
            ctx = ctx.merge(calendario(ctx.ds), on=["unique_id", "ds"])
            fut = calendario(futuro(origem))
            for col, f in colunas_extra.items():
                ctx[col], fut[col] = f(ctx.ds), f(fut.ds)
            fut = fut.drop(columns="mes_ano").rename(columns={"unique_id": "item_id", "ds": "timestamp"})
            ctx = ctx.drop(columns="mes_ano")
        p = chronos.predict_df(ctx.rename(columns={"unique_id": "item_id", "ds": "timestamp", "y": "target"}),
                               future_df=fut, prediction_length=H, quantile_levels=[0.1, 0.5, 0.9])
        saidas.append(pd.DataFrame({"unique_id": "SIN", "ds": futuro(origem), nome: p["predictions"].values,
                                    f"{nome}-lo-80": p["0.1"].values, f"{nome}-hi-80": p["0.9"].values,
                                    "origem": origem}))
    return pd.concat(saidas)

t0 = time.time()
prev_chronos = chronos_backtest()
print(f"Chronos-2: {time.time() - t0:.1f}s")

In [ ]:
import timesfm

tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
tfm.compile(timesfm.ForecastConfig(max_context=512, max_horizon=128, normalize_inputs=True,
                                   use_continuous_quantile_head=True, force_flip_invariance=True,
                                   infer_is_positive=True, fix_quantile_crossing=True))
t0 = time.time()
saidas = []
for origem in ORIGENS:
    historico = serie[serie.ds <= origem].y.values.astype(float)
    ponto, quantis = tfm.forecast(horizon=H, inputs=[historico])
    saidas.append(pd.DataFrame({"unique_id": "SIN", "ds": futuro(origem), "TimesFM-2.5": ponto[0],
                                "TimesFM-2.5-lo-80": quantis[0, :, 1], "TimesFM-2.5-hi-80": quantis[0, :, 9],
                                "origem": origem}))
prev_timesfm = pd.concat(saidas)
print(f"TimesFM 2.5: {time.time() - t0:.1f}s")

## 2.4 Métricas e ranking

In [ ]:
CHAVE = ["unique_id", "ds", "origem"]
backtest = (prev_stats.merge(prev_lgb, on=CHAVE).merge(prev_chronos, on=CHAVE).merge(prev_timesfm, on=CHAVE)
            .merge(serie, on=["unique_id", "ds"]))
MODELOS = ["SeasonalNaive", "AutoETS", "AutoARIMA", "AutoTheta", "LightGBM", "Chronos-2", "TimesFM-2.5"]

def escala_mase(origem):
    hist = serie[serie.ds <= origem].y
    return (hist - hist.shift(12)).abs().mean()

def metricas(bt, modelos):
    linhas = []
    for m in modelos:
        erro = bt[m] - bt.y
        mase = np.mean([(erro[bt.origem == o].abs().mean() / escala_mase(o)) for o in bt.origem.unique()])
        linha = {"modelo": m, "MAE (MWmed)": erro.abs().mean(), "RMSE (MWmed)": np.sqrt((erro ** 2).mean()),
                 "MAPE (%)": (erro.abs() / bt.y).mean() * 100, "MASE": mase, "Viés (MWmed)": erro.mean()}
        if f"{m}-lo-80" in bt:
            dentro = (bt.y >= bt[f"{m}-lo-80"]) & (bt.y <= bt[f"{m}-hi-80"])
            linha["Cobertura 80% (%)"] = dentro.mean() * 100
        linhas.append(linha)
    return pd.DataFrame(linhas).sort_values("MAPE (%)").reset_index(drop=True)

ranking = metricas(backtest, MODELOS)
ranking

In [ ]:
por_origem = pd.DataFrame({f"{o:%m/%Y}": {m: ((backtest[backtest.origem == o][m] - backtest[backtest.origem == o].y).abs()
                                               / backtest[backtest.origem == o].y).mean() * 100 for m in MODELOS}
                           for o in ORIGENS})
print("MAPE (%) por ponto de corte:")
por_origem.round(1)

In [ ]:
fig, eixos = plt.subplots(1, len(ORIGENS), figsize=(16, 4), sharey=True)
destaque = ranking.modelo.head(3).tolist() + (["SeasonalNaive"] if "SeasonalNaive" not in ranking.modelo.head(3).tolist() else [])
for eixo, origem in zip(eixos, ORIGENS):
    bt = backtest[backtest.origem == origem]
    hist = serie[(serie.ds > origem - pd.DateOffset(months=24)) & (serie.ds <= origem)]
    eixo.plot(hist.ds, hist.y / 1000, color="gray", lw=1)
    eixo.plot(bt.ds, bt.y / 1000, color="black", lw=2, label="realizado")
    for m in destaque:
        eixo.plot(bt.ds, bt[m] / 1000, lw=1.3, label=m)
    eixo.set_title(f"Corte em {origem:%m/%Y}"); eixo.tick_params(axis="x", rotation=45)
eixos[0].set_ylabel("mil MWmed"); eixos[-1].legend(fontsize=8); plt.tight_layout(); plt.show()

## 2.5 O erro em reais

Pela política (fictícia) da Aurora Energia, a carteira da empresa equivale a **2% da carga do SIN**. Um erro de previsão
do SIN vira um erro proporcional na carteira, e cada 1 MWmed de diferença durante um mês equivale a cerca de 730 MWh,
liquidados ao PLD de referência de R$ 250/MWh.

In [ ]:
PLD = 250            # R$/MWh, valor de referência hipotético
HORAS_MES = 730
PARTICIPACAO = 0.02  # a carteira da Aurora equivale a 2% da carga do SIN
ranking["Custo anual do erro (R$ mi)"] = ranking["MAE (MWmed)"] * PARTICIPACAO * HORAS_MES * PLD * 12 / 1e6
ranking[["modelo", "MAPE (%)", "MAE (MWmed)", "Custo anual do erro (R$ mi)"]]

## 2.6 Previsão dos próximos 12 meses

In [ ]:
melhor_estatistico = ranking[ranking.modelo.isin(["AutoETS", "AutoARIMA", "AutoTheta"])].modelo.iloc[0]
mod = {"AutoETS": AutoETS, "AutoARIMA": AutoARIMA, "AutoTheta": AutoTheta}[melhor_estatistico](season_length=12)
p_stat = StatsForecast(models=[mod], freq="MS").forecast(df=serie, h=H, level=[80])
p_chr = chronos.predict_df(serie.rename(columns={"unique_id": "item_id", "ds": "timestamp", "y": "target"}),
                           prediction_length=H, quantile_levels=[0.1, 0.5, 0.9])
previsao = pd.DataFrame({
    "mes": futuro(ULTIMO),
    melhor_estatistico: p_stat[melhor_estatistico].values,
    "Chronos-2": p_chr["predictions"].values, "Chronos-2 (p10)": p_chr["0.1"].values, "Chronos-2 (p90)": p_chr["0.9"].values,
})

fig, eixo = plt.subplots(figsize=(14, 4))
hist = serie[serie.ds > ULTIMO - pd.DateOffset(months=36)]
eixo.plot(hist.ds, hist.y / 1000, color="black", label="histórico")
eixo.plot(previsao.mes, previsao[melhor_estatistico] / 1000, label=melhor_estatistico)
eixo.plot(previsao.mes, previsao["Chronos-2"] / 1000, label="Chronos-2")
eixo.fill_between(previsao.mes, previsao["Chronos-2 (p10)"] / 1000, previsao["Chronos-2 (p90)"] / 1000, alpha=0.2, label="Chronos-2, intervalo de 80%")
eixo.set_ylabel("mil MWmed"); eixo.legend(); plt.show()

ranking.to_csv("resultados/ranking.csv", index=False)
backtest.to_csv("resultados/backtest.csv", index=False)
previsao.to_csv("resultados/previsao_12_meses.csv", index=False)
previsao.round(0)

# Parte 3 — Agente analista: LLM + ferramentas + RAG + skills

O agente usa o **Qwen3-4B-Instruct-2507**, um LLM aberto (licença Apache 2.0) baixado do Hugging Face e executado aqui no Colab,
com a biblioteca **smolagents**. Ele recebe uma pergunta, decide quais ferramentas chamar, lê os resultados e responde.

In [ ]:
# A Parte 3 pode rodar sem refazer as Partes 1 e 2: lê os resultados salvos e recalcula o que precisa.
from statsmodels.tsa.seasonal import STL

dados = pd.read_csv("dados/carga_sin_mensal.csv", parse_dates=["mes"]).set_index("mes")
carga = dados["carga_mwmed"]
stl = STL(carga, period=12, robust=True).fit()
comp = pd.DataFrame({"c": carga.pct_change(12), "i": dados["ibc_br"].pct_change(12)}).dropna().loc["2004":"2022"]
comp = comp.drop(pd.date_range("2020-03-01", "2021-06-01", freq="MS"), errors="ignore")
elasticidade = np.polyfit(comp.i, comp.c, 1)[0]
PLD, HORAS_MES, PARTICIPACAO = 250, 730, 0.02

ranking = pd.read_csv("resultados/ranking.csv")
backtest = pd.read_csv("resultados/backtest.csv", parse_dates=["ds", "origem"])
previsao = pd.read_csv("resultados/previsao_12_meses.csv", parse_dates=["mes"])
print("Resultados da Parte 2 carregados:", len(ranking), "modelos,", len(previsao), "meses de previsão")

## 3.1 O LLM

In [ ]:
from smolagents import TransformersModel

MODELO_LLM = "Qwen/Qwen3-0.6B" if SMOKE else "Qwen/Qwen3-4B-Instruct-2507"
llm = TransformersModel(model_id=MODELO_LLM, device_map=DEVICE,
                        torch_dtype="float16" if DEVICE == "cuda" else "float32",
                        max_new_tokens=128 if SMOKE else 1024,
                        apply_chat_template_kwargs={"enable_thinking": False})

def perguntar(texto):
    """Uma pergunta direta ao LLM, sem ferramentas."""
    return llm.generate([{"role": "user", "content": [{"type": "text", "text": texto}]}]).content

print(perguntar("Em uma frase: o que é a carga de energia do Sistema Interligado Nacional?"))

## 3.2 RAG: a base de documentos

A pasta `documentos/` tem textos sobre os eventos que marcaram a série, a política de contratação da Aurora Energia e um glossário.
Cada parágrafo vira um embedding; a busca devolve os parágrafos mais parecidos com a pergunta.

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer

trechos = []
for arquivo in sorted(Path("documentos").glob("*.md")):
    texto = arquivo.read_text()
    titulo = texto.splitlines()[0].lstrip("# ").strip()
    for paragrafo in texto.split("\n\n")[1:]:
        if len(paragrafo.strip()) > 40:
            trechos.append({"fonte": titulo, "texto": paragrafo.strip()})

embedder = SentenceTransformer("intfloat/multilingual-e5-small", device=DEVICE)
E_trechos = embedder.encode([f"passage: {t['fonte']}. {t['texto']}" for t in trechos], normalize_embeddings=True)

def buscar(consulta, k=3):
    e = embedder.encode([f"query: {consulta}"], normalize_embeddings=True)[0]
    ordem = (E_trechos @ e).argsort()[::-1][:k]
    return [trechos[i] | {"similaridade": float(E_trechos[i] @ e)} for i in ordem]

print(len(trechos), "trechos indexados")
for t in buscar("por que a carga caiu em 2020?"):
    print(f"{t['similaridade']:.2f} [{t['fonte']}] {t['texto'][:100]}...")

## 3.3 Skills

Uma *skill* é um procedimento escrito em linguagem natural, guardado em um arquivo `SKILL.md`, com nome e descrição no cabeçalho.
O agente vê só as descrições; quando uma tarefa pede uma skill, ele carrega as instruções completas.

In [ ]:
import re

def ler_skill(caminho):
    texto = caminho.read_text()
    cabecalho = dict(re.findall(r"^(name|description):\s*(.+)$", texto.split("---")[1], re.M))
    return cabecalho["name"], cabecalho["description"], texto.split("---", 2)[2].strip()

SKILLS = {nome: (desc, corpo) for nome, desc, corpo in map(ler_skill, sorted(Path("skills").glob("*/SKILL.md")))}
for nome, (desc, _) in SKILLS.items():
    print(f"- {nome}: {desc}")
print("\n" + Path("skills/relatorio_executivo/SKILL.md").read_text())

## 3.4 As ferramentas do agente

In [ ]:
from smolagents import tool

@tool
def ranking_modelos() -> str:
    """Devolve a tabela de erro de cada modelo no backtest de 12 meses, do melhor para o pior."""
    colunas = ["modelo", "MAPE (%)", "MAE (MWmed)", "Viés (MWmed)", "Custo anual do erro (R$ mi)"]
    return ranking[colunas].round(2).to_string(index=False)

@tool
def previsao_12_meses() -> str:
    """Devolve a previsão de carga (MWmed) dos próximos 12 meses, mês a mês, com o intervalo de 80% do Chronos-2."""
    return previsao.assign(mes=previsao.mes.dt.strftime("%m/%Y")).round(0).to_string(index=False)

@tool
def maiores_desvios(n: int) -> str:
    """Lista os meses em que a carga mais se afastou do padrão de tendência e sazonalidade (resíduo da decomposição STL).

    Args:
        n: quantos meses listar
    """
    r = (stl.resid / carga * 100).round(1)
    top = r.abs().nlargest(int(n)).index
    return "\n".join(f"{m:%m/%Y}: {r[m]:+.1f}% em relação ao esperado" for m in sorted(top))

@tool
def cenario_atividade(variacao_pct: float) -> str:
    """Simula o efeito sobre a carga dos próximos 12 meses de uma variação adicional na atividade econômica (IBC-Br).

    Args:
        variacao_pct: variação adicional do IBC-Br em pontos percentuais, por exemplo 2 ou -1.5
    """
    base = previsao["Chronos-2"].mean()
    efeito = elasticidade * float(variacao_pct) / 100
    return (f"Carga média prevista (base): {base:,.0f} MWmed. Elasticidade histórica: {elasticidade:.2f}. "
            f"Cenário com {variacao_pct:+.1f} p.p. de IBC-Br: {base * (1 + efeito):,.0f} MWmed "
            f"({efeito * 100:+.2f}%, {base * efeito:+,.0f} MWmed).")

@tool
def custo_do_erro(erro_mwmed: float, meses: int) -> str:
    """Calcula o custo financeiro para a Aurora Energia de um erro na previsão da carga do SIN.

    Args:
        erro_mwmed: tamanho do erro na carga do SIN, em MWmed
        meses: por quantos meses o erro dura
    """
    erro_carteira = abs(float(erro_mwmed)) * PARTICIPACAO
    custo = erro_carteira * HORAS_MES * PLD * int(meses)
    return (f"Um erro de {abs(float(erro_mwmed)):,.0f} MWmed na carga do SIN equivale a {erro_carteira:,.0f} MWmed na carteira "
            f"da Aurora (2% do SIN). Por {int(meses)} meses, custa cerca de R$ {custo / 1e6:,.1f} milhões (PLD de R$ {PLD}/MWh).")

@tool
def buscar_documentos(consulta: str) -> str:
    """Busca nos documentos da empresa (eventos históricos, política de contratação, glossário) os trechos mais relevantes.

    Args:
        consulta: o que procurar, em linguagem natural
    """
    return "\n\n".join(f"[{t['fonte']}] {t['texto']}" for t in buscar(consulta))

@tool
def carregar_skill(nome: str) -> str:
    """Carrega as instruções completas de uma skill. Use antes de executar uma tarefa que corresponda a uma skill.

    Args:
        nome: o nome da skill, por exemplo relatorio_executivo
    """
    return SKILLS[nome][1] if nome in SKILLS else f"Skill '{nome}' não existe. Disponíveis: {list(SKILLS)}"

FERRAMENTAS = [ranking_modelos, previsao_12_meses, maiores_desvios, cenario_atividade,
               custo_do_erro, buscar_documentos, carregar_skill]
print(custo_do_erro(1000, 12))

## 3.5 O agente

In [ ]:
from smolagents import ToolCallingAgent

INSTRUCOES = ("Você é o analista de planejamento da Aurora Energia. Responda em português, de forma objetiva, "
              "usando apenas números devolvidos pelas ferramentas.\n"
              "Skills disponíveis (carregue com carregar_skill antes de usar):\n"
              + "\n".join(f"- {nome}: {desc}" for nome, (desc, _) in SKILLS.items()))

agente = ToolCallingAgent(tools=FERRAMENTAS, model=llm, instructions=INSTRUCOES,
                          max_steps=3 if SMOKE else 8, verbosity_level=1)

def perguntar_ao_agente(pergunta):
    try:
        return agente.run(pergunta)
    except Exception as erro:
        return f"O agente não concluiu: {erro}"

resposta = perguntar_ao_agente("Qual modelo teve o menor erro no backtest e qual foi o MAPE?")
print("\nRESPOSTA:", resposta)

In [ ]:
resposta = perguntar_ao_agente("Por que a carga caiu tanto em abril de 2020? Isso deve afetar a previsão?")
print("\nRESPOSTA:", resposta)

In [ ]:
resposta = perguntar_ao_agente("E se a atividade econômica crescer 2 pontos percentuais acima do esperado? Quanto isso custa se não ajustarmos os contratos?")
print("\nRESPOSTA:", resposta)

In [ ]:
memorando = perguntar_ao_agente("Escreva o memorando de recomendação de contratação para a diretoria.")
print("\nMEMORANDO:\n", memorando)

## 3.6 Uma skill nova

Criar uma skill é escrever um arquivo. Abaixo, uma skill que compara modelos estatísticos com modelos de fundação.

In [ ]:
Path("skills/comparar_familias").mkdir(parents=True, exist_ok=True)
Path("skills/comparar_familias/SKILL.md").write_text("""---
name: comparar_familias
description: Compara o desempenho de modelos estatísticos com o de modelos de fundação (transformers) e explica o resultado.
---

# Comparar famílias de modelos

1. Chame ranking_modelos.
2. Separe os modelos em estatísticos (SeasonalNaive, AutoETS, AutoARIMA, AutoTheta), machine learning (LightGBM)
   e modelos de fundação (Chronos-2, TimesFM-2.5).
3. Diga qual família venceu, por quanto (em pontos de MAPE) e dê uma explicação provável em duas frases.
""".replace("\n        ", "\n"))

SKILLS = {nome: (desc, corpo) for nome, desc, corpo in map(ler_skill, sorted(Path("skills").glob("*/SKILL.md")))}
agente = ToolCallingAgent(tools=FERRAMENTAS, model=llm, max_steps=3 if SMOKE else 8, verbosity_level=1,
                          instructions=INSTRUCOES + f"\n- comparar_familias: {SKILLS['comparar_familias'][0]}")
print("\nRESPOSTA:", perguntar_ao_agente("Os modelos de fundação foram melhores que os estatísticos?"))

# Parte 4 — O LLM pode melhorar a previsão?

## 4.1 Do texto para variáveis: o LLM lê os documentos e monta o calendário de eventos

In [ ]:
import json

textos_eventos = "\n\n".join(Path(f).read_text() for f in sorted(Path("documentos").glob("0[1-5]_*.md")))
PEDIDO = ("Leia os textos abaixo sobre eventos que afetaram a carga de energia do Brasil. Para cada evento, devolva "
          "um objeto JSON com: nome, inicio (AAAA-MM), fim (AAAA-MM, ou null se o efeito for permanente) e "
          "tipo ('temporario' ou 'permanente'). Responda SOMENTE com uma lista JSON, sem texto antes ou depois.\n\n"
          + textos_eventos)

EVENTOS_REFERENCIA = [
    {"nome": "racionamento", "inicio": "2001-06", "fim": "2002-02", "tipo": "temporario"},
    {"nome": "crise_2008", "inicio": "2008-10", "fim": "2009-06", "tipo": "temporario"},
    {"nome": "pandemia", "inicio": "2020-04", "fim": "2020-07", "tipo": "temporario"},
    {"nome": "mmgd_metodologia", "inicio": "2023-05", "fim": None, "tipo": "permanente"},
]

bruto = perguntar(PEDIDO)
try:
    eventos_llm = json.loads(bruto[bruto.find("["): bruto.rfind("]") + 1])
    assert all({"nome", "inicio", "tipo"} <= set(e) for e in eventos_llm)
    print("Eventos extraídos pelo LLM:")
except Exception as erro:
    print(f"Não foi possível ler o JSON do LLM ({erro}). Usando a lista de referência.")
    eventos_llm = EVENTOS_REFERENCIA
pd.DataFrame(eventos_llm)

In [ ]:
def variaveis_de_eventos(eventos):
    """Transforma a lista de eventos em funções que geram uma coluna 0/1 para qualquer conjunto de meses."""
    colunas = {}
    for e in eventos:
        try:
            ini = pd.Timestamp(str(e["inicio"])[:7] + "-01")
            fim = pd.Timestamp(str(e["fim"])[:7] + "-01") if e.get("fim") and e.get("tipo") != "permanente" else None
        except Exception:
            continue
        nome = re.sub(r"\W+", "_", str(e["nome"]).lower()).strip("_")
        if fim is None:
            colunas[nome] = lambda ds, ini=ini: (pd.Series(ds).values >= ini).astype(int)
        else:
            colunas[nome] = lambda ds, ini=ini, fim=fim: ((pd.Series(ds).values >= ini) & (pd.Series(ds).values <= fim)).astype(int)
    return colunas

extras = variaveis_de_eventos(eventos_llm)
print("Variáveis criadas:", list(extras))

prev_lgb_ev = lightgbm_backtest(extras).rename(columns={"LightGBM": "LightGBM + eventos"})
prev_chr_ev = chronos_backtest(extras, nome="Chronos-2 + eventos")
bt_ev = backtest.merge(prev_lgb_ev, on=CHAVE).merge(prev_chr_ev, on=CHAVE)
comparacao = metricas(bt_ev, ["LightGBM", "LightGBM + eventos", "Chronos-2", "Chronos-2 + eventos", ranking.modelo.iloc[0]])
comparacao.drop_duplicates("modelo")

## 4.2 O LLM como previsor de números

Um LLM também pode receber a série escrita como texto e "continuar" os números. Vamos comparar com o Chronos-2 no último ponto de corte.

In [ ]:
origem = ORIGENS[-1]
hist = serie[serie.ds <= origem].tail(48)
texto_serie = ", ".join(f"{v / 1000:.1f}" for v in hist.y)
pedido = (f"Esta é a carga mensal de energia do Brasil, em mil MWmed, nos últimos 48 meses, do mais antigo para o mais recente:\n"
          f"{texto_serie}\n\nPreveja os próximos 12 meses. Responda SOMENTE com 12 números separados por vírgula, sem texto.")
bruto = perguntar(pedido)
numeros = [float(x) * 1000 for x in re.findall(r"\d+(?:\.\d+)?", bruto)][:H]
real = serie[serie.ds.isin(futuro(origem))].y.values
chr_ult = backtest[backtest.origem == origem]["Chronos-2"].values

if len(numeros) == H:
    mape_llm = np.mean(np.abs(np.array(numeros) - real) / real) * 100
    print(f"MAPE do LLM como previsor: {mape_llm:.1f}% | Chronos-2: {np.mean(np.abs(chr_ult - real) / real) * 100:.1f}%")
    fig, eixo = plt.subplots(figsize=(12, 4))
    eixo.plot(hist.ds, hist.y / 1000, color="gray")
    eixo.plot(futuro(origem), real / 1000, color="black", lw=2, label="realizado")
    eixo.plot(futuro(origem), np.array(numeros) / 1000, label=f"LLM ({MODELO_LLM.split('/')[-1]})")
    eixo.plot(futuro(origem), chr_ult / 1000, label="Chronos-2")
    eixo.legend(); eixo.set_ylabel("mil MWmed"); plt.show()
else:
    print(f"O LLM devolveu {len(numeros)} números em vez de {H}:\n{bruto[:300]}")

## Resumo

| Etapa | O que foi feito |
|---|---|
| Exploração | Tendência, sazonalidade, quebras, autocorrelação, estacionariedade, relação com o IBC-Br |
| Previsão | 7 modelos em backtest de 12 meses com 3 pontos de corte; ranking por MAPE e custo em R$ |
| Agente | LLM aberto com 7 ferramentas, RAG sobre documentos e skills |
| LLM na previsão | Variáveis de eventos extraídas de texto; LLM como previsor direto |